# Skills básicas del agente

Borrador de trabajo para la Fase 3 del TFM. Las cinco primeras skills son comunes a los agentes que consulten datos o documentos. La sexta es específica del **Macro Agent** y sirve para probar el primer flujo completo.

El punto de partida es una **revisión diaria**. Para cada corte de decisión, un proceso prepara las cotizaciones y series del universo del TFM y enumera todas las publicaciones de las fuentes cubiertas que se hicieron disponibles desde el corte anterior. El agente revisa ese conjunto y explica qué cambió, con citas y avisos de cobertura.

Una **skill** describe cómo revisar e interpretar esas entradas. Las **tools** cargan datos, leen objetos de S3 y aplican el corte temporal. La búsqueda por texto queda como apoyo para localizar pasajes en informes largos. El flujo diario puede empezar sin embeddings.

Este notebook recoge el diseño para desarrollarlo después. El estado de las herramientas y los agentes se describe en el [plan del TFM](../../docs/tfm/README.md) y en su [arquitectura](../../docs/tfm/arquitectura.md).

## 1. Preparar las publicaciones del día desde el catálogo

**Objetivo.** Obtener el inventario completo de las publicaciones disponibles para esa revisión dentro de las fuentes cubiertas.

- Partir del catálogo documental. En el corpus de bancos y riesgo, el punto de entrada documentado en S3 es `catalogo/documentos.csv`.
- Enumerar las publicaciones y versiones que se hicieron disponibles entre el corte anterior y el actual. El catálogo determina qué documentos entran, incluidos los de fines de semana o festivos si el siguiente corte es el de una sesión posterior.
- Registrar la institución, tipo, versión, referencias al texto y al original, y el estado de extracción. Conservar en el inventario los problemas de lectura o cobertura como avisos.

**Salida esperada.** Inventario de las publicaciones del intervalo y avisos sobre fuentes o documentos que no se pudieron cubrir.

[Estructura del corpus en S3 y campos del catálogo](../../docs/fase2/bancos_riesgo/revision_cartera_s3.md)

## 2. Revisar las publicaciones del día

**Objetivo.** Revisar cada publicación del inventario y extraer las novedades que importan para el análisis.

- Leer el texto extraído de cada documento. En informes largos, revisar por secciones y resumir cada documento antes de integrar sus hallazgos en la revisión diaria.
- Usar búsqueda textual dentro de los documentos cuando ayude a localizar un dato o un pasaje concreto. La práctica de [retrieval de 10-K](https://github.com/Romequinco/10k-financial-agent/blob/codigo/docs/11_skill_mejora_retrieval.md) sirve de referencia para esa búsqueda auxiliar.
- Abrir el PDF o HTML original mediante una tool cuando el hallazgo dependa de una tabla, un gráfico o un pasaje con extracción dudosa.
- Si el pasaje sigue siendo ilegible o falta en el corpus, registrar la limitación concreta.

**Salida esperada.** Estado de revisión por documento y novedades relevantes con su localización en el original.

## 3. Trabajar a fecha de decisión

**Objetivo.** Reconstruir qué información estaba disponible cuando se tomó la decisión.

- Distinguir la fecha del evento, la fecha de publicación y la disponibilidad acreditada de la versión concreta. Por ejemplo, unas actas solo pueden entrar en el análisis desde su publicación, aunque describan una reunión anterior.
- Usar el corte que fija el sistema. El **TimeGate** de cada tool aplica ese corte antes de devolver documentos o datos. En el corpus documental, el modo acreditado exige evidencia de la versión disponible. En modo observado, la copia puede usarse desde su primera observación.
- Aplicar el mismo corte a publicaciones, cotizaciones y series. Una publicación que llega después del corte pasa a la revisión siguiente. Las publicaciones previas al intervalo se pueden consultar como contexto si eran elegibles a esa fecha.

**Salida esperada.** Fecha de decisión y evidencia admisible para esa fecha, con los avisos relevantes.

[Contrato de herramientas](../../src/herramientas/README.md)

## 4. Consultar series y régimen

**Objetivo.** Revisar cómo se movió el universo de mercado cubierto y qué datos macro estaban disponibles al corte.

- Cargar las cotizaciones y series del universo definido. Comparar con la observación previa cuando proceda y distinguir un dato nuevo de la última observación disponible. Comprobar unidades, frecuencia y versión al corte para series revisables.
- Consultar `get_regimen(fecha)` cuando corresponda al rol. Leer el estado, la fecha efectiva del dato y los avisos devueltos.
- Usar el régimen para decidir qué evidencia merece atención. Por ejemplo, en una crisis puede ser más relevante examinar el crédito y las medidas de liquidez.

**Salida esperada.** Contexto de mercado breve, con el régimen y las observaciones que ayudan a interpretar el hallazgo.

## 5. Entregar hallazgos con citas

**Objetivo.** Dejar una salida que otro agente o una persona pueda revisar y reutilizar.

- Vincular cada hallazgo con su evidencia, guardando la fuente, la versión y la página, sección o fragmento cuando corresponda. En una serie, identificar la observación que sustenta el hallazgo.
- Distinguir lo que afirma la fuente de la interpretación del agente. Si dos fuentes discrepan, indicar en qué discrepan y citar ambas.

**Salida esperada.** Revisión diaria con fecha de decisión, movimientos de mercado, novedades documentales, citas y límites de cobertura.

## 6. Comparar comunicados de bancos centrales

**Objetivo.** Dar al Macro Agent un procedimiento para explicar qué cambió entre dos comunicados consecutivos.

1. Si hay un comunicado nuevo en el inventario diario, localizar el anterior de la misma institución y del mismo tipo que esté disponible al corte.
2. Comparar la decisión de política monetaria, el diagnóstico económico y el lenguaje sobre decisiones futuras.
3. Citar pasajes de ambas publicaciones para justificar cada cambio relevante. Cuando la evidencia lo permita, explicar si el tono se volvió más restrictivo, más expansivo o se mantuvo.
4. Añadir el contexto de series y régimen que ayude a interpretar el cambio. Si falta el comunicado anterior, indicar que la comparación queda pendiente.

**Salida esperada.** Un resumen de qué cambió y por qué importa, con citas y los datos de contexto disponibles al corte.

## Primera prueba para retomar el trabajo

> Al corte de X, ¿qué se publicó desde la última revisión, qué se movió en el universo de mercado y qué información merece destacarse?

1. Elegir un corte y preparar un caso con publicaciones de varias entidades, al menos una posterior al corte y cotizaciones del universo cubierto.
2. Comprobar que cada publicación elegible figura en el inventario y se revisa o queda marcada con su incidencia. Verificar los movimientos de mercado y una cita contra su fuente.
3. Avanzar al corte siguiente y comprobar que la publicación posterior entra entonces. Probar también una jornada sin publicaciones nuevas para verificar que la revisión de mercado se sigue entregando.